# XGBoost + GridSearch — Skenario A-Gabungan
**Data:** Gabungan → Train:56.108 | Test:14.027  
**Referensi:** Chimphlee et al. (2024); Elgeldawi et al. (2021); Hafizah et al. (2025)  
**CV:** StratifiedKFold n=5, scoring=f1_macro — Atsauri et al. (2023)


## Cell 1 — Setup

In [6]:
import pandas as pd, numpy as np, os, pickle, time, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
SPLIT_DIR  = os.path.join(BASE_DIR, 'splits')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [MODEL_DIR, RESULT_DIR]: os.makedirs(d, exist_ok=True)

SEED=42; LABEL_MAP={'keluhan':0,'saran':1,'pujian':2}
INV_MAP={v:k for k,v in LABEL_MAP.items()}; CLASS_NAMES=['keluhan','saran','pujian']

def evaluate(y_true, y_pred, exp_code, prefix):
    acc=accuracy_score(y_true,y_pred)
    mac=f1_score(y_true,y_pred,average='macro',zero_division=0)
    f1s=f1_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    prec=precision_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rec=recall_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rep=classification_report(y_true,y_pred,target_names=CLASS_NAMES,digits=4,zero_division=0)
    print(f'\n{"="*60}'); print(f'HASIL — {exp_code}'); print(f'{"="*60}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Macro F1   : {mac:.4f}')
    print(f'  F1 Keluhan : {f1s[0]:.4f}  Prec:{prec[0]:.4f}  Rec:{rec[0]:.4f}')
    print(f'  F1 Saran   : {f1s[1]:.4f}  Prec:{prec[1]:.4f}  Rec:{rec[1]:.4f}')
    print(f'  F1 Pujian  : {f1s[2]:.4f}  Prec:{prec[2]:.4f}  Rec:{rec[2]:.4f}')
    print(f'\n{rep}')
    cm=confusion_matrix(y_true,y_pred,labels=[0,1,2])
    fig,ax=plt.subplots(figsize=(6,5)); vmax=cm.max()
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=vmax)
    for i in range(3):
        for j in range(3):
            c='white' if cm[i,j]>vmax*0.55 else '#1A1A1A'
            ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',fontsize=12,fontweight='bold',color=c)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted',fontweight='bold'); ax.set_ylabel('Actual',fontweight='bold')
    ax.set_title(f'{exp_code}\nAcc={acc*100:.2f}% | MacroF1={mac:.4f}',fontweight='bold',pad=10)
    plt.colorbar(im,ax=ax,shrink=0.85); fig.tight_layout()
    path=os.path.join(RESULT_DIR,f'CM_{prefix}.png')
    fig.savefig(path); plt.close(); print(f'  CM → {path}')
    return {'exp':exp_code,'accuracy':acc,'macro_f1':mac,
            'f1_keluhan':f1s[0],'f1_saran':f1s[1],'f1_pujian':f1s[2]}

def tfidf_chi2(X_train, X_test, y_train):
    tfidf=TfidfVectorizer(ngram_range=(1,1),max_features=50000,
                          sublinear_tf=True,min_df=2,strip_accents='unicode')
    Xt=tfidf.fit_transform(X_train); Xte=tfidf.transform(X_test)
    k=min(1500,Xt.shape[1]); sel=SelectKBest(chi2,k=k)
    Xts=sel.fit_transform(Xt,y_train); Xtes=sel.transform(Xte)
    print(f'TF-IDF: {Xt.shape[1]:,} → Chi2: {k:,} fitur')
    return Xts, Xtes, tfidf, sel

print('Setup selesai!')


Setup selesai!


## Cell 2 — Load Data

In [7]:
df_train = pd.read_csv(os.path.join(SPLIT_DIR,'A_gabungan_xgb_train.csv'))
df_test  = pd.read_csv(os.path.join(SPLIT_DIR,'A_gabungan_xgb_test.csv'))
df_train['label_enc']=df_train['label_pks'].map(LABEL_MAP)
df_test['label_enc'] =df_test['label_pks'].map(LABEL_MAP)
X_train=df_train['text'].fillna('').values; y_train=df_train['label_enc'].values
X_test =df_test['text'].fillna('').values;  y_test =df_test['label_enc'].values
print(f'Skenario: A-Gabungan | Train:{len(X_train):,} | Test:{len(X_test):,}')
for l,c in zip(*np.unique(y_train,return_counts=True)):
    print(f'  {INV_MAP[l]:10s}: {c:,} ({c/len(y_train)*100:.1f}%)')

Skenario: A-Gabungan | Train:45,052 | Test:11,263
  keluhan   : 31,135 (69.1%)
  saran     : 6,317 (14.0%)
  pujian    : 7,600 (16.9%)


## Cell 3 — TF-IDF + Chi-Square

In [8]:
Xtr_sel,Xte_sel,tfidf,sel=tfidf_chi2(X_train,X_test,y_train)

TF-IDF: 4,487 → Chi2: 1,500 fitur


## Cell 4 — Sample Weight

In [9]:
sw=compute_sample_weight(class_weight='balanced',y=y_train)
if 'confidence' in df_train.columns:
    sw=sw*np.clip(df_train['confidence'].values,0.5,1.0)
    print('SW: class weight × confidence')
else:
    print('SW: class weight saja')
print(f'  min={sw.min():.4f} max={sw.max():.4f} mean={sw.mean():.4f}')

SW: class weight × confidence
  min=0.4100 max=2.3773 mean=0.9705


## Cell 5 — GridSearchCV (~30-60 menit)

In [10]:
# ── Cell 5: GridSearch — CPU ──────────────────────────────────────
import xgboost as xgb

param_grid = {
    'n_estimators'    : [300, 500],
    'max_depth'       : [4, 6, 8],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

n_comb = 2*3*3*2*2
print(f'{n_comb} kombinasi × 5 fold = {n_comb*5} fits')

base_xgb = xgb.XGBClassifier(
    objective        = 'multi:softmax',
    num_class        = 3,
    eval_metric      = 'mlogloss',
    device           = 'cpu',
    random_state     = SEED,
    n_jobs           = -1,
    use_label_encoder= False
)

cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(
    base_xgb, param_grid,
    cv      = cv,
    scoring = 'f1_macro',
    n_jobs  = -1,
    verbose = 1,
    refit   = False
)

t0 = time.time()
grid.fit(Xtr_sel, y_train, sample_weight=sw)
print(f'Selesai: {(time.time()-t0)/60:.1f} menit')
print(f'Best params : {grid.best_params_}')
print(f'Best CV F1  : {grid.best_score_:.4f}')

pd.DataFrame(grid.cv_results_).to_csv(
    os.path.join(RESULT_DIR, 'cv_results_XGB_Tuned_A_Gabungan.csv'), index=False
)

72 kombinasi × 5 fold = 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Selesai: 75.6 menit
Best params : {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 500, 'subsample': 1.0}
Best CV F1  : 0.8214


## Cell 6 — Retrain Best Params + GPU

In [11]:
# ── Cell 6: Retrain Best Model — CPU ─────────────────────────────
best_model = xgb.XGBClassifier(
    objective        = 'multi:softmax',
    num_class        = 3,
    eval_metric      = 'mlogloss',
    device           = 'cpu',
    random_state     = SEED,
    n_jobs           = -1,
    use_label_encoder= False,
    **grid.best_params_
)

t0 = time.time()
best_model.fit(
    Xtr_sel, y_train,
    sample_weight = sw,
    eval_set      = [(Xte_sel, y_test)],
    verbose       = False
)
print(f'Retrain: CPU ✅')
print(f'Waktu  : {time.time()-t0:.1f} detik')

y_pred = best_model.predict(Xte_sel)
result = evaluate(y_test, y_pred, 'XGB-Tuned-A-Gabungan', 'XGB-Tuned-A-Gabungan')

pickle.dump(best_model,
    open(os.path.join(MODEL_DIR, 'XGB_Tuned_A_Gabungan.pkl'), 'wb'))
pickle.dump({'tfidf': tfidf, 'selector': sel},
    open(os.path.join(MODEL_DIR, 'vec_XGB_Tuned_A_Gabungan.pkl'), 'wb'))
print('Model tersimpan!')

Retrain: CPU ✅
Waktu  : 44.3 detik

HASIL — XGB-Tuned-A-Gabungan
  Accuracy   : 82.27%
  Macro F1   : 0.7750
  F1 Keluhan : 0.8793  Prec:0.9404  Rec:0.8257
  F1 Saran   : 0.5923  Prec:0.4817  Rec:0.7688
  F1 Pujian  : 0.8532  Prec:0.8512  Rec:0.8553

              precision    recall  f1-score   support

     keluhan     0.9404    0.8257    0.8793      7784
       saran     0.4817    0.7688    0.5923      1579
      pujian     0.8512    0.8553    0.8532      1900

    accuracy                         0.8227     11263
   macro avg     0.7578    0.8166    0.7750     11263
weighted avg     0.8611    0.8227    0.8347     11263

  CM → C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\CM_XGB-Tuned-A-Gabungan.png
Model tersimpan!
